# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities are referenced by their `@id` fields as per Croissant best practices.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL (Croissant JSON-LD schema URL)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m\n{metadata.description}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s in the dataset.

> **Note:** All entities (record sets, fields/columns) are referenced by their `@id`.

In [ ]:
# Retrieve and display the available record sets
record_set_ids = [record_set['@id'] for record_set in metadata.to_json().get('recordSet', [])]

if not record_set_ids:
    raise ValueError("No record sets found in the dataset metadata.")

print("Available Record Sets (by @id):")
for id_ in record_set_ids:
    print(f"  - {id_}")

# For each record set, print out its fields/columns and their @ids
print("\nFields/columns in each Record Set:")
for record_set in metadata.to_json().get('recordSet', []):
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"\nRecord Set @id: {record_set['@id']}")
    for f in fields:
        f_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
        print(f"   - Field/Column @id: {f_id}")

## 3. Data Extraction
Let's fetch all data for each record set as a pandas DataFrame, referencing record set and field/column by their `@id`.

In [ ]:
# Compose a list of all record set @ids
record_sets = record_set_ids

# Dictionary for holding all DataFrames by record set @id
dataframes = {}

for rset_id in record_sets:
    # Each record is dict mapping field @id to value
    records = list(dataset.records(record_set=rset_id))
    if len(records) == 0:
        print(f"No records found for record set: {rset_id}")
    else:
        df = pd.DataFrame(records)
        print(f"Loaded DataFrame for Record Set {rset_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(2))
        dataframes[rset_id] = df

# Pick the first available record set for downstream examples
example_record_set_id = record_sets[0]
example_df = dataframes.get(example_record_set_id, pd.DataFrame())

print(f"\nFirst few rows for record set {example_record_set_id}:")
print(example_df.head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate EDA by selecting a numeric field for analysis, filtering based on a condition, normalizing the field, and grouping by a categorical field. All references are via field or record set `@id`.

In [ ]:
# --- Edit these to match your dataset field @id ---
# Below is an illustrative example. Replace with actual field @ids from your data overview if needed.
df = example_df

# Try to pick a numeric column; fall back to first column if none
sample_numeric_id = None
for col in df.columns:
    try:
        # Heuristically check if column is numeric by sampling values
        vals = pd.to_numeric(df[col], errors='coerce')
        if vals.notnull().mean() > 0.7:
            sample_numeric_id = col
            break
    except Exception:
        continue
if sample_numeric_id is None:
    # Fallback to the first column
    sample_numeric_id = df.columns[0]

print(f"Selected numeric field @id: {sample_numeric_id}")

# Convert column to numeric, ignore errors
df[sample_numeric_id] = pd.to_numeric(df[sample_numeric_id], errors='coerce')

# Remove records where value is missing
filtered_df = df[df[sample_numeric_id].notnull()]

# Use a threshold (mean) and create filtered view
threshold = filtered_df[sample_numeric_id].mean() if len(filtered_df) > 0 else 0
filtered_df = filtered_df[filtered_df[sample_numeric_id] > threshold]
print(f"Filtered records with {sample_numeric_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
if filtered_df[sample_numeric_id].std() > 0:
    filtered_df[f"{sample_numeric_id}_normalized"] = (
        (filtered_df[sample_numeric_id] - filtered_df[sample_numeric_id].mean()) / filtered_df[sample_numeric_id].std()
    )
    print(f"\nNormalized {sample_numeric_id} for filtered records:")
    print(filtered_df[[sample_numeric_id, f"{sample_numeric_id}_normalized"]].head())

# Try to find a suitable group field (categorical)
group_field = None
for col in df.columns:
    if col == sample_numeric_id:
        continue
    # Heuristically treat as group field if < 10 unique values and non-numeric
    if df[col].dtype == object and df[col].nunique() < 10:
        group_field = col
        break
if group_field:
    print(f"\nGrouping by field @id: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[sample_numeric_id].mean()
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship with a group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(filtered_df) > 0:
    ax = filtered_df[sample_numeric_id].plot(kind='hist', bins=15, alpha=0.6, color='skyblue')
    plt.title(f"Distribution of {sample_numeric_id}")
    plt.xlabel(sample_numeric_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[sample_numeric_id])
        plt.title(f"{sample_numeric_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(sample_numeric_id)
        plt.show()

## 6. Conclusion

- We successfully loaded the FAIR² dataset using `mlcroissant` and referenced all structures via `@id` fields.
- Explored the available record sets and their columns.
- Performed basic filtering, normalization, and grouping, and visualized numeric data distributions.
- For more in-depth analysis, consult the full Croissant schema to select specific fields and apply domain-specific operations.

This demonstrates a general workflow for robust and reproducible FAIR data exploration!